# MuseTalk coarse lip-sync en Colab

Executa les cel·les en ordre en un runtime nou amb GPU T4. Si una cel·la d'instal·lació o descàrrega falla, atura't i copia l'error complet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import shutil
from pathlib import Path
repo_dir = Path('/content/lipsync-pipeline')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
!git clone --recurse-submodules https://github.com/arnaumartin10/SmartDub.git /content/lipsync-pipeline
%cd /content/lipsync-pipeline
!git submodule update --init --recursive
print('Repository cloned with submodules.')

In [ ]:
!nvidia-smi
!ffmpeg -version | head -n 2
import torch
print(f'Torch preinstalled: {torch.__version__}; CUDA: {torch.version.cuda}; available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CUDA is unavailable. Stop and copy the complete error.'

## Instal·lació

Aquesta cel·la conserva el Torch/CUDA preinstal·lat per Colab. No instal·la `requirements.txt` complet perquè això podria substituir el runtime CUDA. No instal·la `opencv-python`: només s'utilitza `opencv-contrib-python`, compatible amb NumPy 2 i MediaPipe.

In [ ]:
%cd /content/lipsync-pipeline
import importlib.metadata as metadata
import subprocess
import sys

def pip_install(*packages, no_deps=False):
    command = [sys.executable, '-m', 'pip', 'install', '--upgrade', '-q']
    if no_deps:
        command.append('--no-deps')
    subprocess.check_call(command + list(packages))

# Installs package code without allowing pip to replace Colab's Torch/CUDA.
pip_install(
    'diffusers==0.30.2', 'accelerate==0.28.0',
    'transformers==4.48.3', 'huggingface_hub==0.36.2',
    'whisperx==3.8.6', 'faster-whisper==1.2.0', 'ctranslate2==4.8.2',
    'g2p_en==2.1.0', 'mediapipe==0.10.35', 'scenedetect==0.6.7',
    'soundfile==0.12.1', 'librosa==0.10.2.post1', 'einops==0.8.1',
    'omegaconf', 'ffmpeg-python', 'opencv-contrib-python==5.0.0.93',
    no_deps=True,
)

# Runtime dependencies. These packages do not install Torch.
pip_install(
    'onnxruntime>=1.20,<2', 'pyannote-audio==4.0.0', 'torchcodec==0.7.0',
    'av', 'pandas', 'scipy', 'tqdm', 'pyyaml', 'sentencepiece',
    'inflect==7.5.0', 'distance==0.1.3',
)

# Fail before model/checkpoint cells if the installed runtime is incomplete.
import numpy as np
import cv2
import mediapipe
import scenedetect
import onnxruntime
import faster_whisper
import whisperx
import g2p_en

print(f'Torch: {torch.__version__}; CUDA: {torch.version.cuda}; available: {torch.cuda.is_available()}')
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('MediaPipe:', mediapipe.__version__)
print('PySceneDetect:', scenedetect.__version__)
print('WhisperX:', metadata.version('whisperx'))
print('CTranslate2:', metadata.version('ctranslate2'))
print('ONNX Runtime:', metadata.version('onnxruntime'))
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0), 'NumPy 2.x is required.'
assert tuple(int(x) for x in cv2.__version__.split('.')[:2]) >= (5, 0), 'OpenCV 5.x is required.'
assert mediapipe.__version__ == '0.10.35', f'Unexpected MediaPipe: {mediapipe.__version__}'
assert metadata.version('whisperx') == '3.8.6'
assert metadata.version('ctranslate2') == '4.8.2'
print('Imports: cv2, mediapipe, scenedetect, whisperx, faster_whisper, onnxruntime, g2p_en OK')
print('Dependency installation completed successfully.')

In [ ]:
%cd /content/lipsync-pipeline
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '-q', '--no-deps', 'huggingface_hub==0.36.2', 'gdown'])
for directory in ('models/musetalkV15', 'models/sd-vae', 'models/whisper'):
    Path(directory).mkdir(parents=True, exist_ok=True)

# Modern MediaPipe Tasks needs this face-landmarker model; legacy Solutions does not.
face_model = Path('models/face_landmarker.task')
if not face_model.is_file():
    subprocess.check_call([
        'wget', '-q', '--show-progress', '-O', str(face_model),
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task',
    ])

def hf_download(repo, local_dir, *patterns):
    subprocess.check_call([
        'hf', 'download', repo, '--local-dir', local_dir,
        '--include', *patterns,
    ])

hf_download('TMElyralab/MuseTalk', 'models', 'musetalkV15/musetalk.json', 'musetalkV15/unet.pth')
hf_download('stabilityai/sd-vae-ft-mse', 'models/sd-vae', 'config.json', 'diffusion_pytorch_model.bin')
hf_download('openai/whisper-tiny', 'models/whisper', 'config.json', 'pytorch_model.bin', 'preprocessor_config.json')

required = [
    Path('models/face_landmarker.task'),
    Path('models/musetalkV15/musetalk.json'),
    Path('models/musetalkV15/unet.pth'),
    Path('models/sd-vae/config.json'),
    Path('models/sd-vae/diffusion_pytorch_model.bin'),
    Path('models/whisper/config.json'),
    Path('models/whisper/pytorch_model.bin'),
    Path('models/whisper/preprocessor_config.json'),
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Checkpoint download failed; missing or empty files: {missing}')
print('Download completed. Checkpoint files:')
for path in required:
    print(f'  {path}: {path.stat().st_size} bytes')

In [ ]:
%cd /content/lipsync-pipeline
from pathlib import Path

DRIVE_INPUT = Path('/content/drive/MyDrive/SmartDub/data/inputs')
Path('data/inputs').mkdir(parents=True, exist_ok=True)
required_inputs = ['sample.mp4', 'sample_dub.wav', 'sample_dub.txt']
missing = [name for name in required_inputs if not (DRIVE_INPUT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing Drive inputs in {DRIVE_INPUT}: {missing}')
for name in required_inputs:
    target = Path('data/inputs') / name
    target.write_bytes((DRIVE_INPUT / name).read_bytes())
print('Input files copied:')
for name in required_inputs:
    path = Path('data/inputs') / name
    print(f'  {path}: {path.stat().st_size} bytes')
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames -of default=noprint_wrappers=1 data/inputs/sample.mp4
!ffprobe -v error -select_streams a:0 -show_entries stream=sample_rate,channels,duration -of default=noprint_wrappers=1 data/inputs/sample_dub.wav

In [ ]:
%cd /content/lipsync-pipeline
from pathlib import Path
from src.generation.coarse_lipsync import CoarseLipSyncGenerator

assert Path('third_party/MuseTalk').is_dir()
assert Path('models/face_landmarker.task').is_file()
assert Path('models/musetalkV15/unet.pth').is_file()
assert Path('models/sd-vae/config.json').is_file()
assert Path('models/whisper/config.json').is_file()
print('Repository, checkpoints, MediaPipe model, and wrapper paths are ready.')

In [ ]:
%cd /content/lipsync-pipeline
!python scripts/generate_demo.py \
  --video data/inputs/sample.mp4 \
  --audio data/inputs/sample_dub.wav \
  --transcript data/inputs/sample_dub.txt \
  --checkpoint-dir models \
  --output data/outputs/musetalk_coarse.mp4

In [ ]:
%cd /content/lipsync-pipeline
from pathlib import Path
output = Path('data/outputs/musetalk_coarse.mp4')
assert output.is_file() and output.stat().st_size > 0, 'MuseTalk output was not created.'
!ffprobe -v error -show_entries format=duration -show_entries stream=width,height,r_frame_rate,nb_frames -of default=noprint_wrappers=1 data/outputs/musetalk_coarse.mp4
drive_output = Path('/content/drive/MyDrive/SmartDub/data/outputs')
drive_output.mkdir(parents=True, exist_ok=True)
(drive_output / output.name).write_bytes(output.read_bytes())
print(f'Copied result to {drive_output / output.name}')